In [1]:
import importlib
import os
import shutil
import kagglehub
import pandas as pd

# Reset kagglehub in case dataset_download was overwritten as a string
importlib.reload(kagglehub)

# Download both datasets
path1 = kagglehub.dataset_download(
    "sumansharmadataworld/depression-surveydataset-for-analysis"
)
path2 = kagglehub.dataset_download(
    "vishalfalnikar/mental-health-dataset-machine-learning"
)

print("Files in path1:", os.listdir(path1))
print("Files in path2:", os.listdir(path2))

# Copy raw CSV files into ml/datasets/
os.makedirs("../datasets", exist_ok=True)
for p in [path1, path2]:
    for file in os.listdir(p):
        if file.endswith(".csv"):
            shutil.copy(os.path.join(p, file), os.path.join("../datasets", file))

# Load the CSV from path1 into a Pandas DataFrame
csv_file1 = [f for f in os.listdir(path1) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path1, csv_file1))

print(f"\nLoaded dataset shape: {df.shape}")
print(df.head())

Extracting files...
Extracting files...
Files in path1: ['final_depression_dataset_1.csv']
Files in path2: ['Mental_health - Mental_health.csv']

Loaded dataset shape: (2556, 19)
      Name  Gender  Age       City Working Professional or Student  \
0    Pooja  Female   37  Ghaziabad            Working Professional   
1  Reyansh    Male   60     Kalyan            Working Professional   
2    Manvi  Female   42     Bhopal            Working Professional   
3     Isha  Female   44      Thane            Working Professional   
4    Aarav    Male   48     Indore            Working Professional   

          Profession  Academic Pressure  Work Pressure  CGPA  \
0            Teacher                NaN            2.0   NaN   
1  Financial Analyst                NaN            4.0   NaN   
2            Teacher                NaN            2.0   NaN   
3            Teacher                NaN            3.0   NaN   
4     UX/UI Designer                NaN            4.0   NaN   

   Study Satisf

In [2]:
import os
import pandas as pd

# 1. Load CSVs from both downloaded paths
csv1 = [
    f for f in os.listdir(path1) if f.endswith(".csv") or f.endswith(".DATA")
][0]
csv2 = [
    f for f in os.listdir(path2) if f.endswith(".csv") or f.endswith(".DATA")
][0]

df1 = pd.read_csv(os.path.join(path1, csv1))
df2 = pd.read_csv(os.path.join(path2, csv2))

print(f"Dataset 1 Shape: {df1.shape}")
print(f"Dataset 2 Shape: {df2.shape}")

# 2. Align and Concatenate Both Datasets
common_columns = list(set(df1.columns).intersection(set(df2.columns)))
print(f"Matching features across both sets: {len(common_columns)}")

# Merge into one master training dataset
master_df = pd.concat([df1[common_columns], df2[common_columns]], ignore_index=True)
print(f"Combined Master Dataset Shape: {master_df.shape}")

# 3. Data Cleaning (Handling Noise & Missing Values)
master_df.drop_duplicates(inplace=True)

print("\nMissing values per column before cleaning:")
print(master_df.isnull().sum())

# Replace string question marks ('?') or malformed placeholders with NaN
master_df.replace("?", pd.NA, inplace=True)

# Fill numeric missing values with median, categorical with mode
for col in master_df.select_dtypes(include=["number"]).columns:
    master_df[col] = master_df[col].fillna(master_df[col].median())

for col in master_df.select_dtypes(include=["object"]).columns:
    if not master_df[col].mode().empty:
        master_df[col] = master_df[col].fillna(master_df[col].mode()[0])

print("\nMissing values after cleaning:", master_df.isnull().sum().sum())

# Save cleaned dataset
os.makedirs("../datasets", exist_ok=True)
cleaned_path = "../datasets/cleaned_master_mental_health_data.csv"
master_df.to_csv(cleaned_path, index=False)
print(f"Cleaned master dataset saved to '{cleaned_path}'!")

Dataset 1 Shape: (2556, 19)
Dataset 2 Shape: (140700, 20)
Matching features across both sets: 19
Combined Master Dataset Shape: (143256, 19)

Missing values per column before cleaning:
Financial Stress                              4
Have you ever had suicidal thoughts ?         0
Study Satisfaction                       114857
Academic Pressure                        114857
City                                          0
Profession                                37303
Gender                                        0
Work Pressure                             28420
CGPA                                     114856
Age                                           0
Working Professional or Student               0
Job Satisfaction                          28412
Dietary Habits                                4
Work/Study Hours                              0
Sleep Duration                                0
Degree                                        2
Name                                          0

In [3]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# 1. Encode categorical text features and identifiers from master_df
print(f"Loaded Dataset Shape: {master_df.shape}")
df_encoded = master_df.copy()
if "Name" in df_encoded.columns:
    df_encoded.drop(columns=["Name"], inplace=True)

for col in df_encoded.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# 2. Define Features (X) and Target (y)
target_column = "Depression" if "Depression" in df_encoded.columns else df_encoded.columns[-1]
X = df_encoded.drop(columns=[target_column])
y = df_encoded[target_column]

if y.dtype == "object":
    le_y = LabelEncoder()
    y = le_y.fit_transform(y.astype(str))

print(f"Features shape (X): {X.shape}, Target shape (y): {y.shape}")

# 3. Train / Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Model Training: Random Forest vs XGBoost
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=8, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        eval_metric="mlogloss",
        random_state=42,
    ),
}

best_model = None
best_score = 0

for name, clf in models.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"{name} Accuracy: {acc * 100:.2f}%")
    print(classification_report(y_test, preds, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

    if acc > best_score:
        best_score = acc
        best_model = clf

# 5. Save Trained Pipeline Artifact
os.makedirs("../models", exist_ok=True)
model_path = "../models/hridya_screening_model.joblib"
joblib.dump(best_model, model_path)
print(f"\nSaved Best Model ({best_model.__class__.__name__}) to '{model_path}'!")

Loaded Dataset Shape: (143256, 19)
Features shape (X): (143256, 17), Target shape (y): (143256,)

Training Random Forest...
Random Forest Accuracy: 91.69%
              precision    recall  f1-score   support

           0       0.94      0.97      0.95     23027
           1       0.83      0.79      0.81      5114
           2       0.00      0.00      0.00       420
           3       0.00      0.00      0.00        91

    accuracy                           0.92     28652
   macro avg       0.44      0.44      0.44     28652
weighted avg       0.90      0.92      0.91     28652

Confusion Matrix:
 [[22251   776     0     0]
 [ 1093  4021     0     0]
 [  413     7     0     0]
 [   25    66     0     0]]

Training XGBoost...
XGBoost Accuracy: 91.95%
              precision    recall  f1-score   support

           0       0.94      0.97      0.95     23027
           1       0.82      0.81      0.81      5114
           2       0.00      0.00      0.00       420
           3       

In [4]:
import joblib
import numpy as np

# 1. Load the trained model artifact
model = joblib.load("../models/hridya_screening_model.joblib")
print(f"Loaded model type: {model.__class__.__name__}")

# 2. Run inference on a sample record from X_test
sample_input = X_test.iloc[[0]]
prediction = int(model.predict(sample_input)[0])
probabilities = model.predict_proba(sample_input)[0]
confidence = float(np.max(probabilities)) * 100

print(f"\nTest Result:")
print(f"Predicted Class Code: {prediction}")
print(f"Confidence Score: {confidence:.2f}%")

Loaded model type: XGBClassifier

Test Result:
Predicted Class Code: 0
Confidence Score: 96.42%
